In [9]:
import pandas as pd
file_path = "NarrateIQ_ML_Dataset.xlsx"
df = pd.read_excel(file_path)
df.head()

,Student_ID,Age_Group,Disability_Type,Severity_Level,State,School_Type,Language,Teacher_Experience,Input_Mode,Content_Format,...,Offline_Mode_Used,Device_Type,Parental_Involvement,Engagement_Score,Attention_Span_Min,Completion_Rate_Pct,Comprehension_Score,Teacher_Satisfaction,Improvement_Score_Pct,Learning_Outcome
0,STU1000,11-13,Autism,Mild,UP,Special School,Hindi,5+ Years,Video,Animated Video,...,0,Laptop,3,63.4,6.9,86.4,42.2,4.4,46.6,Average
1,STU1001,5-7,Multiple,Mild,UP,NGO,Telugu,3-5 Years,Video,Text+Audio,...,1,Phone,2,79.0,21.8,61.9,68.5,3.9,30.2,Average
2,STU1002,5-7,ADHD,Mild,Karnataka,Government,Hindi,3-5 Years,Audio,Animated Video,...,0,Tablet,2,63.4,3.7,66.5,61.4,4.1,32.6,Average
3,STU1003,8-10,ADHD,Mild,West Bengal,Government,English,1-3 Years,Text,ISL Avatar,...,0,Phone,4,79.6,6.0,69.0,86.6,4.4,72.6,Good
4,STU1004,8-10,Dyslexia,Mild,Gujarat,Government,Kannada,1-3 Years,Video,Animated Video,...,1,Phone,4,82.7,13.8,100.0,83.4,3.1,42.6,Good


In [10]:
import numpy as np
import pandas as pd

rename_map = {'test_score(out of 10)': 'test_score',
              'interview_score(out of 10)': 'interview_score',
              'salary($)': 'salary'}
present_keys = [k for k in rename_map.keys() if k in df.columns]
if present_keys:
    df.rename(columns={k: rename_map[k] for k in present_keys}, inplace=True)
else:
    print('No expected columns found to rename; continuing.')

def word_to_num(word):
    word_dict = {'one':1, 'two':2, 'three':3, 'four':4, 'five':5, 'six':6, 'seven':7, 'eight':8,
                'nine':9, 'ten':10, 'eleven':11, 'zero':0, 'nan':np.nan}
    return word_dict.get(str(word).lower(), word)
if 'experience' in df.columns:
    df['experience'] = df['experience'].apply(word_to_num)
    df['experience'] = df['experience'].fillna(0)
else:
    print('Column `experience` not found; creating with zeros.')
    df['experience'] = 0

for col in ['test_score', 'interview_score']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
    else:
        print(f'Column `{col}` not found; skipping conversion.')
df.info()
display(df.head())

No expected columns found to rename; continuing.
Column `experience` not found; creating with zeros.
Column `test_score` not found; skipping conversion.
Column `interview_score` not found; skipping conversion.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             600 non-null    object 
 1   Age_Group              600 non-null    object 
 2   Disability_Type        600 non-null    object 
 3   Severity_Level         600 non-null    object 
 4   State                  600 non-null    object 
 5   School_Type            600 non-null    object 
 6   Language               600 non-null    object 
 7   Teacher_Experience     600 non-null    object 
 8   Input_Mode             600 non-null    object 
 9   Content_Format         600 non-null    object 
 10  ISL_Used               600 non-null    int64  
 11  Sess

,Student_ID,Age_Group,Disability_Type,Severity_Level,State,School_Type,Language,Teacher_Experience,Input_Mode,Content_Format,...,Device_Type,Parental_Involvement,Engagement_Score,Attention_Span_Min,Completion_Rate_Pct,Comprehension_Score,Teacher_Satisfaction,Improvement_Score_Pct,Learning_Outcome,experience
0,STU1000,11-13,Autism,Mild,UP,Special School,Hindi,5+ Years,Video,Animated Video,...,Laptop,3,63.4,6.9,86.4,42.2,4.4,46.6,Average,0
1,STU1001,5-7,Multiple,Mild,UP,NGO,Telugu,3-5 Years,Video,Text+Audio,...,Phone,2,79.0,21.8,61.9,68.5,3.9,30.2,Average,0
2,STU1002,5-7,ADHD,Mild,Karnataka,Government,Hindi,3-5 Years,Audio,Animated Video,...,Tablet,2,63.4,3.7,66.5,61.4,4.1,32.6,Average,0
3,STU1003,8-10,ADHD,Mild,West Bengal,Government,English,1-3 Years,Text,ISL Avatar,...,Phone,4,79.6,6.0,69.0,86.6,4.4,72.6,Good,0
4,STU1004,8-10,Dyslexia,Mild,Gujarat,Government,Kannada,1-3 Years,Video,Animated Video,...,Phone,4,82.7,13.8,100.0,83.4,3.1,42.6,Good,0


In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

required_cols = ['experience', 'test_score', 'interview_score']
for c in required_cols:
    if c not in df.columns:
        print(f'Column {c} missing — creating default zeros.')
        df[c] = 0
# Prefer `salary` if present, otherwise fall back to `Improvement_Score_Pct`
if 'salary' in df.columns:
    target_col = 'salary'
elif 'Improvement_Score_Pct' in df.columns:
    target_col = 'Improvement_Score_Pct'
    print('Using `Improvement_Score_Pct` as target since `salary` not found.')
else:
    raise KeyError('Neither `salary` nor `Improvement_Score_Pct` found in dataframe')
X = df[required_cols]
y = df[target_col]
# Drop rows with missing target
mask = y.notnull()
X = X[mask]
y = y[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
print('Model training complete.')
print(f'Target column: {target_col}')
print(f'Model coefficients: {model.coef_}')
print(f'Model intercept: {model.intercept_}')
print(f'R^2 train: {r2_score(y_train, model.predict(X_train)):.4f}')
print(f'R^2 test: {r2_score(y_test, model.predict(X_test)):.4f}')


Using `Improvement_Score_Pct` as target since `salary` not found.
Model training complete.
Target column: Improvement_Score_Pct
Model coefficients: [0. 0. 0.]
Model intercept: 39.37229166666667
R^2 train: 0.0000
R^2 test: -0.0041


In [16]:
scenario1 = pd.DataFrame([[2, 9, 6]], columns=['experience', 'test_score', 'interview_score'])
scenario2 = pd.DataFrame([[12, 10, 10]], columns=['experience', 'test_score', 'interview_score'])

prediction1 = model.predict(scenario1)[0]
prediction2 = model.predict(scenario2)[0]

print(f"Predicted {target_col} for 2 years experience, 9 test score, 6 interview score: {prediction1:.2f}")
print(f"Predicted {target_col} for 12 years experience, 10 test score, 10 interview score: {prediction2:.2f}")

Predicted Improvement_Score_Pct for 2 years experience, 9 test score, 6 interview score: 39.37
Predicted Improvement_Score_Pct for 12 years experience, 10 test score, 10 interview score: 39.37
